In [ ]:
# @title Cell 1: Setup & Imports (Run Once)
# Single pip install for all dependencies
!pip install yfinance selenium webdriver-manager --quiet

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
import io
import os
from datetime import datetime, timedelta

# Colab-specific imports (wrapped in try for local testing)
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("All imports loaded successfully!")

In [ ]:
# @title Cell 2: Configuration Constants

# Israeli CPI Inflation Data (Source: CBS Israel)
INFLATION_DATA = {
    2004: 1.2, 2005: 2.4, 2006: -0.1, 2007: 3.4, 2008: 3.8, 2009: 3.9,
    2010: 2.7, 2011: 2.2, 2012: 1.6, 2013: 1.8, 2014: -0.2,
    2015: -1.0, 2016: -0.2, 2017: 0.4, 2018: 0.8, 2019: 0.6,
    2020: -0.7, 2021: 2.8, 2022: 5.3, 2023: 3.0, 2024: 3.2,
    2025: 2.4
}

# Portfolio Definitions
PORTFOLIOS = {
    'Golden Butterfly': ['VTI', 'VBR', 'TLT', 'SHY', 'GLD'],
    'Ivy Portfolio': ['VTI', 'VEU', 'AGG', 'DBC', 'VNQ'],
    'S&P 500': ['IVV'],
    'Nasdaq 100': ['QQQ'],
    'Semiconductors': ['SOXX']
}

# Asset mappings for single-asset analysis
ASSETS = {
    'Silver': 'SLV',
    'Gold': 'IAU',
    'S&P 500': 'IVV',
    'Nasdaq': 'QQQ',
    'Semiconductors': 'SMH'
}

# Currency pair for USD/ILS
USD_ILS_TICKER = 'ILS=X'

# Trading days per year
TRADING_DAYS = 252

print("Configuration loaded!")

In [ ]:
# @title Cell 3: Core Utility Functions

def download_market_data(tickers, start_date='2005-01-01', period=None):
    """
    Download market data from Yahoo Finance.
    Uses vectorized operations for efficiency.
    """
    # Ensure tickers is a list
    if isinstance(tickers, str):
        tickers = [tickers]
    
    # Always include USD/ILS for currency conversion
    all_tickers = list(set(tickers + [USD_ILS_TICKER]))
    
    print(f"Downloading data for {len(all_tickers)} tickers...")
    
    if period:
        data = yf.download(all_tickers, period=period, progress=False)['Close']
    else:
        data = yf.download(all_tickers, start=start_date, progress=False)['Close']
    
    # Clean data: forward fill then drop remaining NaN
    data = data.ffill().dropna()
    
    return data


def build_cpi_index(date_index, inflation_data=None, base_value=100.0):
    """
    Build daily CPI index from annual inflation rates.
    Uses vectorized interpolation for efficiency.
    """
    if inflation_data is None:
        inflation_data = INFLATION_DATA
    
    # Initialize series with NaN
    cpi_daily = pd.Series(index=date_index, dtype=float)
    cpi_daily.iloc[0] = base_value
    current_cpi = base_value
    
    # Set CPI values at year-end dates
    for year in sorted(inflation_data.keys()):
        if year >= date_index[0].year:
            rate = inflation_data[year] / 100
            current_cpi *= (1 + rate)
            year_days = date_index[date_index.year == year]
            if len(year_days) > 0:
                cpi_daily.loc[year_days[-1]] = current_cpi
    
    # Interpolate linearly between year-end points
    cpi_daily = cpi_daily.interpolate(method='time').ffill()
    
    return cpi_daily


def convert_to_ils(data, usd_ils_col=None):
    """
    Convert USD prices to ILS using vectorized multiplication.
    """
    if usd_ils_col is None:
        usd_ils = data[USD_ILS_TICKER]
    else:
        usd_ils = usd_ils_col
    
    # Vectorized conversion: multiply all columns by USD/ILS rate
    data_ils = data.drop(columns=[USD_ILS_TICKER], errors='ignore')
    data_ils = data_ils.multiply(usd_ils, axis=0)
    
    return data_ils, usd_ils


def calculate_log_returns(prices):
    """
    Calculate log returns using NumPy for efficiency.
    Log returns are more accurate for cumulative calculations.
    """
    if isinstance(prices, pd.DataFrame):
        return np.log(prices / prices.shift(1)).dropna()
    else:
        # NumPy array
        return np.diff(np.log(prices))


def calculate_cumulative_returns(log_returns, base=0):
    """
    Calculate cumulative returns from log returns.
    Uses vectorized cumsum for efficiency.
    """
    if isinstance(log_returns, pd.Series):
        cumulative = np.exp(log_returns.cumsum()) - 1
    else:
        # NumPy array
        cumulative = np.exp(np.cumsum(log_returns)) - 1
        cumulative = np.insert(cumulative, 0, base)
    
    return cumulative


def calculate_portfolio_returns(returns_df, tickers, weights=None):
    """
    Calculate portfolio returns with equal or custom weights.
    Uses vectorized operations.
    """
    if weights is None:
        # Equal weight
        return returns_df[tickers].mean(axis=1)
    else:
        # Weighted returns using dot product
        weights = np.array(weights)
        return returns_df[tickers].dot(weights)


def calculate_volatility(returns, annualize=True):
    """
    Calculate annualized volatility.
    """
    vol = np.std(returns)
    if annualize:
        vol *= np.sqrt(TRADING_DAYS)
    return vol * 100


print("Core utility functions loaded!")

In [ ]:
# @title Cell 4: Plotting Utility Functions

def plot_cumulative_returns(dates, series_dict, title, ylabel='Cumulative Return (%)'):
    """
    Generic function to plot multiple cumulative return series.
    """
    plt.figure(figsize=(12, 6))
    
    for label, (data, style) in series_dict.items():
        color = style.get('color', 'blue')
        linestyle = style.get('linestyle', '-')
        linewidth = style.get('linewidth', 1.5)
        plt.plot(dates, data * 100, label=label, color=color, 
                 linestyle=linestyle, linewidth=linewidth)
    
    plt.title(title)
    plt.xlabel('Date')
    plt.ylabel(ylabel)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_normalized_prices(prices_df, title, base=100):
    """
    Plot normalized prices (all starting at base value).
    """
    normalized = (prices_df / prices_df.iloc[0]) * base
    
    colors = {
        'S&P 500': 'blue',
        'Nasdaq 100': 'black',
        'Semiconductors': 'purple',
        'Golden Butterfly': '#D4AF37',
        'Ivy Portfolio': 'green'
    }
    
    plt.figure(figsize=(14, 8))
    
    for col in normalized.columns:
        plt.plot(normalized.index, normalized[col],
                 label=col, color=colors.get(col, 'gray'), linewidth=2)
    
    plt.axhline(y=base, color='gray', linestyle='--', linewidth=1)
    plt.title(title, fontsize=16)
    plt.ylabel(f'Normalized Value (Base={base})')
    plt.xlabel('Year')
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


print("Plotting utilities loaded!")

In [ ]:
# @title Cell 5: Single Asset Analysis (Silver, Gold, S&P 500)

def analyze_asset_returns(asset_ticker, asset_name, period='1y'):
    """
    Analyze returns for a single asset in USD vs ILS.
    """
    tickers = [asset_ticker, USD_ILS_TICKER]
    
    print(f"Downloading data for {asset_name}...")
    data = yf.download(tickers, period=period, progress=False)['Close']
    data = data.ffill().dropna()
    
    # Convert to NumPy for vectorized calculations
    asset_usd = data[asset_ticker].to_numpy()
    usd_ils = data[USD_ILS_TICKER].to_numpy()
    
    # Vectorized ILS conversion
    asset_ils = np.multiply(asset_usd, usd_ils)
    
    # Vectorized log returns
    log_ret_usd = np.diff(np.log(asset_usd))
    log_ret_ils = np.diff(np.log(asset_ils))
    
    # Vectorized cumulative returns
    cum_usd = np.exp(np.cumsum(log_ret_usd)) - 1
    cum_ils = np.exp(np.cumsum(log_ret_ils)) - 1
    
    # Insert zero at start
    cum_usd = np.insert(cum_usd, 0, 0)
    cum_ils = np.insert(cum_ils, 0, 0)
    
    # Statistics
    vol_usd = np.std(log_ret_usd) * np.sqrt(TRADING_DAYS) * 100
    vol_ils = np.std(log_ret_ils) * np.sqrt(TRADING_DAYS) * 100
    correlation = np.corrcoef(log_ret_usd, log_ret_ils)[0, 1]
    gap = (cum_ils[-1] - cum_usd[-1]) * 100
    
    # Print results
    print(f"\n--- {asset_name} Analysis Results ---")
    print(f"Annual Return (USD): {cum_usd[-1]*100:.2f}%")
    print(f"Annual Return (ILS Adjusted): {cum_ils[-1]*100:.2f}%")
    print(f"Currency Impact: {gap:+.2f}%")
    print(f"Volatility (USD): {vol_usd:.2f}%")
    print(f"Volatility (ILS): {vol_ils:.2f}%")
    print(f"Correlation: {correlation:.4f}")
    
    # Plot
    dates = data.index
    plt.figure(figsize=(12, 6))
    plt.plot(dates, cum_usd * 100, label=f'{asset_name} (USD)', color='blue', linewidth=1.5)
    plt.plot(dates, cum_ils * 100, label=f'{asset_name} (ILS)', color='green', linestyle='--', linewidth=1.5)
    plt.title(f'{asset_name}: USD vs ILS Adjusted Returns')
    plt.xlabel('Date')
    plt.ylabel('Cumulative Return (%)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    return {'cum_usd': cum_usd, 'cum_ils': cum_ils, 'dates': dates}


# Run analyses
def analyze_silver_returns():
    return analyze_asset_returns('SLV', 'Silver (SLV)', '1y')

def analyze_gold_returns_5y():
    return analyze_asset_returns('IAU', 'Gold (IAU)', '5y')

def analyze_sp500_returns_5y():
    return analyze_asset_returns('IVV', 'S&P 500 (IVV)', '1y')

# Example usage:
# analyze_silver_returns()
# analyze_gold_returns_5y()
# analyze_sp500_returns_5y()

In [ ]:
# @title Cell 6: S&P 500 Synthetic Israeli Fund Simulation

def analyze_sp500_synthetic_il():
    """
    Simulate Israeli S&P 500 fund (unhedged) performance.
    """
    tickers = ['IVV', USD_ILS_TICKER]
    
    print("Downloading 5-year data (S&P 500 + USD/ILS)...")
    data = yf.download(tickers, period='5y', progress=False)['Close']
    data = data.ffill().dropna()
    
    if data.empty:
        print("Error: No data received.")
        return
    
    # Vectorized calculations
    sp500_usd = data['IVV'].to_numpy()
    usd_ils = data[USD_ILS_TICKER].to_numpy()
    
    # Synthetic Israeli fund = S&P 500 * USD/ILS
    sp500_ils_synthetic = np.multiply(sp500_usd, usd_ils)
    
    # Log returns
    log_ret_usd = np.diff(np.log(sp500_usd))
    log_ret_ils = np.diff(np.log(sp500_ils_synthetic))
    
    # Cumulative returns
    cum_ret_usd = np.exp(np.cumsum(log_ret_usd)) - 1
    cum_ret_ils = np.exp(np.cumsum(log_ret_ils)) - 1
    cum_ret_usd = np.insert(cum_ret_usd, 0, 0)
    cum_ret_ils = np.insert(cum_ret_ils, 0, 0)
    
    # Statistics
    vol_usd = np.std(log_ret_usd) * np.sqrt(TRADING_DAYS) * 100
    vol_ils = np.std(log_ret_ils) * np.sqrt(TRADING_DAYS) * 100
    gap = (cum_ret_ils[-1] - cum_ret_usd[-1]) * 100
    
    print(f"\n--- S&P 500: USD vs ILS (5 Years) ---")
    print(f"Cumulative Return (USD): {cum_ret_usd[-1]*100:.2f}%")
    print(f"Cumulative Return (ILS - Israeli Fund Simulation): {cum_ret_ils[-1]*100:.2f}%")
    print(f"Currency Impact: {gap:+.2f}%")
    print(f"Volatility (USD): {vol_usd:.2f}%")
    print(f"Volatility (ILS): {vol_ils:.2f}%")
    
    # Plot
    dates = data.index
    plt.figure(figsize=(12, 6))
    plt.plot(dates, cum_ret_usd * 100, label='S&P 500 (USD)', color='blue', linewidth=1.5)
    plt.plot(dates, cum_ret_ils * 100, label='S&P 500 (ILS - Synthetic)', color='green', linestyle='--', linewidth=1.5)
    plt.title('S&P 500: USD vs Israeli Fund Simulation (5 Years)')
    plt.xlabel('Date')
    plt.ylabel('Cumulative Return (%)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# analyze_sp500_synthetic_il()

In [ ]:
# @title Cell 7: TASE API Data Fetching

def get_tase_history(security_id, years=5):
    """
    Fetch historical data from Tel Aviv Stock Exchange API.
    """
    url = "https://api.tase.co.il/api/Chart/History"
    today = datetime.now()
    start_date = today - timedelta(days=years * 365)
    
    params = {
        "pType": 1,
        "id": security_id,
        "l": 4,
        "d": 0
    }
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Referer": "https://www.tase.co.il/",
        "Accept-Language": "he-IL,he;q=0.9,en-US;q=0.8,en;q=0.7"
    }
    
    print(f"Connecting to TASE for security {security_id}...")
    
    try:
        response = requests.get(url, params=params, headers=headers, timeout=30)
        response.raise_for_status()
        data = response.json()
        
        if 'c' in data:
            df = pd.DataFrame(data['c'])
            df = df[['d', 'c']]
            df.columns = ['Date', 'Close']
            
            # Parse TASE date format
            df['Date'] = df['Date'].astype(str).str.extract(r'(\d+)').astype(int)
            df['Date'] = pd.to_datetime(df['Date'], unit='ms')
            df.set_index('Date', inplace=True)
            df = df[df.index >= start_date]
            
            print(f"Retrieved {len(df)} data points.")
            return df
        else:
            print("No data in response.")
            return None
            
    except Exception as e:
        print(f"Error fetching TASE data: {e}")
        return None

# Example: get_tase_history(1159250)

In [ ]:
# @title Cell 8: Selenium Web Scraping Setup (Optional)

def setup_selenium_colab():
    """
    Setup Chrome for Selenium in Google Colab.
    Run this only if you need web scraping.
    """
    # Clean install
    os.system("apt-get remove -y chromium-browser chromium-chromedriver 2>/dev/null")
    os.system("apt-get update -qq")
    os.system("apt-get install -y wget unzip -qq")
    
    print("Installing Google Chrome...")
    os.system("wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb")
    os.system("apt-get install -y ./google-chrome-stable_current_amd64.deb -qq")
    
    print("Chrome installed successfully!")


def get_chrome_driver():
    """
    Create a headless Chrome driver for web scraping.
    """
    from selenium import webdriver
    from selenium.webdriver.chrome.service import Service
    from selenium.webdriver.chrome.options import Options
    from webdriver_manager.chrome import ChromeDriverManager
    
    chrome_options = Options()
    chrome_options.add_argument('--headless')
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.add_argument('--disable-blink-features=AutomationControlled')
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=chrome_options
    )
    return driver


def scrape_funder_price(fund_id):
    """
    Scrape current price from Funder.co.il
    """
    import time
    
    print(f"Fetching data for fund {fund_id}...")
    
    try:
        driver = get_chrome_driver()
        url = f"https://www.funder.co.il/fund/{fund_id}"
        driver.get(url)
        time.sleep(3)
        
        title = driver.title
        print(f"Connected to: {title}")
        
        body_text = driver.find_element("tag name", "body").text
        if "שער אחרון" in body_text:
            print("Page loaded successfully!")
        
        driver.quit()
        return True
        
    except Exception as e:
        print(f"Error: {e}")
        return False

# Run setup_selenium_colab() first if needed
# scrape_funder_price("1159250")

In [ ]:
# @title Cell 9: CSV File Upload Analysis

def analyze_uploaded_fund(filter_last_year=False):
    """
    Analyze uploaded CSV file with fund data.
    Compare with S&P 500.
    """
    if not IN_COLAB:
        print("This function requires Google Colab.")
        return
    
    print("Step 1: Please select your CSV file...")
    uploaded = colab_files.upload()
    
    if not uploaded:
        print("No file uploaded.")
        return
    
    file_name = next(iter(uploaded))
    print(f"File '{file_name}' loaded successfully!")
    
    try:
        df = pd.read_csv(io.BytesIO(uploaded[file_name]))
    except Exception as e:
        print(f"Error reading file: {e}")
        return
    
    # Detect column names (Hebrew or English)
    if 'תאריך' in df.columns:
        date_col, price_col = 'תאריך', 'מחיר פדיון'
    elif 'Date' in df.columns:
        date_col, price_col = 'Date', 'Price'
    else:
        date_col, price_col = df.columns[1], df.columns[3]
    
    df = df.rename(columns={date_col: 'Date', price_col: 'IL_Price'})
    df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
    df = df.dropna(subset=['Date']).sort_values('Date').set_index('Date')
    
    # Clean price column
    if df['IL_Price'].dtype == object:
        df['IL_Price'] = df['IL_Price'].astype(str).str.replace(',', '').astype(float)
    
    # Filter to last year if requested
    if filter_last_year:
        last_date = df.index[-1]
        start_date = last_date - timedelta(days=365)
        df = df[df.index >= start_date]
        print(f"Filtered to: {df.index[0].date()} to {df.index[-1].date()}")
    
    # Calculate returns
    df['IL_Return'] = (df['IL_Price'] / df['IL_Price'].iloc[0] - 1) * 100
    
    # Download S&P 500 for comparison
    start_date = df.index[0]
    end_date = df.index[-1] + timedelta(days=5)
    
    print(f"Downloading S&P 500 data for comparison...")
    us_data = yf.download('IVV', start=start_date, end=end_date, progress=False)['Close']
    
    # Merge datasets
    combined = pd.merge_asof(
        df[['IL_Return']], 
        us_data.to_frame('US_Price'),
        left_index=True, 
        right_index=True,
        direction='nearest'
    )
    combined['US_Return'] = (combined['US_Price'] / combined['US_Price'].iloc[0] - 1) * 100
    
    # Plot
    plt.figure(figsize=(12, 6))
    plt.plot(combined.index, combined['IL_Return'], label='Israeli Fund', color='green', linewidth=1.5)
    plt.plot(combined.index, combined['US_Return'], label='S&P 500 (USD)', color='blue', linestyle='--', linewidth=1.5)
    plt.title('Israeli Fund vs S&P 500')
    plt.xlabel('Date')
    plt.ylabel('Return (%)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"\nFinal Returns:")
    print(f"Israeli Fund: {combined['IL_Return'].iloc[-1]:.2f}%")
    print(f"S&P 500: {combined['US_Return'].iloc[-1]:.2f}%")

# analyze_uploaded_fund()
# analyze_uploaded_fund(filter_last_year=True)  # For last year only

In [ ]:
# @title Cell 10: Portfolio Comparison Engine (Core)

def build_portfolio_prices(data, portfolio_tickers, usd_ils):
    """
    Build portfolio price series in ILS.
    Uses vectorized operations for efficiency.
    """
    if len(portfolio_tickers) == 1:
        # Single asset portfolio
        return data[portfolio_tickers[0]] * usd_ils
    else:
        # Multi-asset portfolio with equal weights
        basket = data[portfolio_tickers].multiply(usd_ils, axis=0)
        returns = basket.pct_change()
        portfolio_returns = returns.mean(axis=1)
        # Build price series from returns
        prices = (1 + portfolio_returns).cumprod()
        prices.iloc[0] = 1  # Start at 1
        return prices * 100  # Scale to 100


def compare_portfolios(start_year='2007', portfolios_to_compare=None, include_inflation=True):
    """
    Compare multiple portfolios with optional inflation adjustment.
    """
    if portfolios_to_compare is None:
        portfolios_to_compare = ['Golden Butterfly', 'Ivy Portfolio']
    
    # Collect all tickers needed
    all_tickers = [USD_ILS_TICKER]
    for name in portfolios_to_compare:
        all_tickers.extend(PORTFOLIOS[name])
    all_tickers = list(set(all_tickers))
    
    print(f"Downloading historical data from {start_year}...")
    data = yf.download(all_tickers, start=f"{start_year}-01-01", progress=False)['Close']
    data = data.ffill().dropna()
    
    if data.empty:
        print("Error: No data received.")
        return None
    
    usd_ils = data[USD_ILS_TICKER]
    
    # Build CPI index if needed
    if include_inflation:
        cpi = build_cpi_index(data.index)
    
    # Calculate portfolio values
    results = pd.DataFrame(index=data.index)
    
    for name in portfolios_to_compare:
        tickers = PORTFOLIOS[name]
        
        if len(tickers) == 1:
            # Single asset
            prices_ils = data[tickers[0]] * usd_ils
            nom_returns = (prices_ils / prices_ils.iloc[0]) * 100
        else:
            # Multi-asset portfolio
            basket_ils = data[tickers].multiply(usd_ils, axis=0)
            daily_returns = basket_ils.pct_change().mean(axis=1)
            nom_returns = (1 + daily_returns).cumprod() * 100
        
        results[f'{name} (Nominal)'] = nom_returns
        
        if include_inflation:
            # Real returns = Nominal / CPI * 100
            results[f'{name} (Real)'] = nom_returns / cpi * 100
    
    if include_inflation:
        results['CPI Index'] = cpi
    
    return results, data

print("Portfolio comparison engine loaded!")

In [ ]:
# @title Cell 11: Golden Butterfly vs Ivy Portfolio (20 Years)

def compare_portfolios_shekel_20y():
    """
    Compare Golden Butterfly vs Ivy Portfolio over ~20 years in ILS.
    """
    golden_tickers = PORTFOLIOS['Golden Butterfly']
    ivy_tickers = PORTFOLIOS['Ivy Portfolio']
    
    all_tickers = list(set(golden_tickers + ivy_tickers + [USD_ILS_TICKER]))
    
    print("Downloading 20 years of historical data (starting 2006)...")
    data = yf.download(all_tickers, start="2006-01-01", progress=False)['Close']
    data = data.ffill().dropna()
    
    # Convert to ILS
    usd_ils = data[USD_ILS_TICKER]
    data_ils = data.drop(columns=[USD_ILS_TICKER]).multiply(usd_ils, axis=0)
    
    # Calculate daily returns
    returns = data_ils.pct_change().dropna()
    
    # Portfolio returns (equal weight)
    golden_daily = returns[golden_tickers].mean(axis=1)
    ivy_daily = returns[ivy_tickers].mean(axis=1)
    
    # Cumulative returns
    golden_cum = (1 + golden_daily).cumprod() * 100
    ivy_cum = (1 + ivy_daily).cumprod() * 100
    
    # Statistics
    print(f"\n--- Portfolio Comparison (ILS, ~20 Years) ---")
    print(f"Golden Butterfly Final Value: {golden_cum.iloc[-1]:.2f} (Return: {golden_cum.iloc[-1]-100:.2f}%)")
    print(f"Ivy Portfolio Final Value: {ivy_cum.iloc[-1]:.2f} (Return: {ivy_cum.iloc[-1]-100:.2f}%)")
    
    # Plot
    plt.figure(figsize=(14, 7))
    plt.plot(golden_cum.index, golden_cum, label='Golden Butterfly', color='#D4AF37', linewidth=2)
    plt.plot(ivy_cum.index, ivy_cum, label='Ivy Portfolio', color='green', linewidth=2)
    plt.axhline(y=100, color='gray', linestyle='--', linewidth=1)
    plt.title('Golden Butterfly vs Ivy Portfolio (ILS, 2006-Present)', fontsize=14)
    plt.xlabel('Date')
    plt.ylabel('Portfolio Value (Base=100)')
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    return golden_cum, ivy_cum

# compare_portfolios_shekel_20y()

In [ ]:
# @title Cell 12: Portfolio Comparison with Inflation Adjustment

def compare_portfolios_with_inflation():
    """
    Compare portfolios with Israeli CPI inflation adjustment.
    Shows both nominal and real returns.
    """
    golden_tickers = PORTFOLIOS['Golden Butterfly']
    ivy_tickers = PORTFOLIOS['Ivy Portfolio']
    
    all_tickers = list(set(golden_tickers + ivy_tickers + [USD_ILS_TICKER]))
    
    print("1. Downloading historical data (stocks, bonds, gold, commodities, FX)...")
    data = yf.download(all_tickers, start="2007-01-01", progress=False)['Close']
    data = data.ffill().dropna()
    
    # Convert to ILS
    usd_ils = data[USD_ILS_TICKER]
    data_ils = pd.DataFrame(index=data.index)
    for ticker in all_tickers:
        if ticker != USD_ILS_TICKER:
            data_ils[ticker] = data[ticker] * usd_ils
    
    # Daily returns
    returns_ils = data_ils.pct_change().dropna()
    
    # Portfolio returns
    golden_daily = returns_ils[golden_tickers].mean(axis=1)
    ivy_daily = returns_ils[ivy_tickers].mean(axis=1)
    
    # Nominal cumulative
    golden_cum = (1 + golden_daily).cumprod() * 100
    ivy_cum = (1 + ivy_daily).cumprod() * 100
    
    # Build CPI index
    cpi = build_cpi_index(golden_cum.index)
    
    # Real returns (inflation-adjusted)
    golden_real = golden_cum / cpi * 100
    ivy_real = ivy_cum / cpi * 100
    
    # Print results
    print(f"\n--- Results (2007-Present) ---")
    print(f"Golden Butterfly Nominal: {golden_cum.iloc[-1]-100:.2f}%")
    print(f"Golden Butterfly Real: {golden_real.iloc[-1]-100:.2f}%")
    print(f"Ivy Portfolio Nominal: {ivy_cum.iloc[-1]-100:.2f}%")
    print(f"Ivy Portfolio Real: {ivy_real.iloc[-1]-100:.2f}%")
    print(f"Total Inflation: {cpi.iloc[-1]-100:.2f}%")
    
    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Nominal
    ax1.plot(golden_cum.index, golden_cum, label='Golden Butterfly', color='#D4AF37', linewidth=2)
    ax1.plot(ivy_cum.index, ivy_cum, label='Ivy Portfolio', color='green', linewidth=2)
    ax1.plot(cpi.index, cpi, label='CPI (Inflation)', color='red', linestyle='--', linewidth=1.5)
    ax1.axhline(y=100, color='gray', linestyle='--', linewidth=1)
    ax1.set_title('Nominal Returns (ILS)', fontsize=12)
    ax1.set_ylabel('Value (Base=100)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Real
    ax2.plot(golden_real.index, golden_real, label='Golden Butterfly (Real)', color='#D4AF37', linewidth=2)
    ax2.plot(ivy_real.index, ivy_real, label='Ivy Portfolio (Real)', color='green', linewidth=2)
    ax2.axhline(y=100, color='gray', linestyle='--', linewidth=1)
    ax2.set_title('Real Returns (Inflation-Adjusted)', fontsize=12)
    ax2.set_ylabel('Value (Base=100)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return {'golden_nom': golden_cum, 'ivy_nom': ivy_cum, 
            'golden_real': golden_real, 'ivy_real': ivy_real, 'cpi': cpi}

# compare_portfolios_with_inflation()

In [ ]:
# @title Cell 13: Multi-Asset Comparison (Golden Butterfly vs S&P vs QQQ vs Semiconductors)

def compare_all_portfolios(start_year='2015'):
    """
    Compare Golden Butterfly vs S&P 500 vs QQQ vs Semiconductors.
    All returns in ILS with inflation adjustment.
    """
    portfolios = {
        'Golden Butterfly': PORTFOLIOS['Golden Butterfly'],
        'S&P 500': PORTFOLIOS['S&P 500'],
        'Nasdaq 100': PORTFOLIOS['Nasdaq 100'],
        'Semiconductors': ['SMH']
    }
    
    # Collect all tickers
    all_tickers = [USD_ILS_TICKER]
    for tickers in portfolios.values():
        all_tickers.extend(tickers)
    all_tickers = list(set(all_tickers))
    
    print(f"Downloading data from {start_year}...")
    data = yf.download(all_tickers, start=f"{start_year}-01-01", progress=False)['Close']
    data = data.ffill().dropna()
    
    usd_ils = data[USD_ILS_TICKER]
    
    # Build CPI
    cpi = build_cpi_index(data.index)
    
    # Calculate portfolio values
    results = pd.DataFrame(index=data.index)
    
    for name, tickers in portfolios.items():
        if len(tickers) == 1:
            prices_ils = data[tickers[0]] * usd_ils
            nom_returns = (prices_ils / prices_ils.iloc[0]) * 100
        else:
            basket_ils = data[tickers].multiply(usd_ils, axis=0)
            daily_rets = basket_ils.pct_change().mean(axis=1)
            nom_returns = (1 + daily_rets).cumprod() * 100
        
        results[name] = nom_returns
    
    # Real returns
    results_real = results.div(cpi, axis=0) * 100
    
    # Print final values
    print(f"\n--- Final Returns ({start_year}-Present) ---")
    for col in results.columns:
        nom_ret = results[col].iloc[-1] - 100
        real_ret = results_real[col].iloc[-1] - 100
        print(f"{col}: Nominal {nom_ret:.2f}% | Real {real_ret:.2f}%")
    
    # Plot
    colors = {
        'Golden Butterfly': '#D4AF37',
        'S&P 500': 'blue',
        'Nasdaq 100': 'black',
        'Semiconductors': 'purple'
    }
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    for col in results.columns:
        ax1.plot(results.index, results[col], label=col, color=colors.get(col, 'gray'), linewidth=2)
        ax2.plot(results_real.index, results_real[col], label=col, color=colors.get(col, 'gray'), linewidth=2)
    
    ax1.axhline(y=100, color='gray', linestyle='--', linewidth=1)
    ax1.set_title(f'Nominal Returns (ILS) - {start_year} to Present', fontsize=12)
    ax1.set_ylabel('Value (Base=100)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    ax2.axhline(y=100, color='gray', linestyle='--', linewidth=1)
    ax2.set_title(f'Real Returns (Inflation-Adjusted) - {start_year} to Present', fontsize=12)
    ax2.set_ylabel('Value (Base=100)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return results, results_real

# compare_all_portfolios('2015')
# compare_all_portfolios('2005')  # For longer history

In [ ]:
# @title Cell 14: DCA (Dollar Cost Averaging) Simulation with Inflation Adjustment

def simulate_dca(monthly_investment=100, initial_investment=50000, start_year='2005'):
    """
    Simulate Dollar Cost Averaging with inflation-adjusted contributions.
    Monthly investment is adjusted for inflation to maintain purchasing power.
    """
    portfolios = {
        'Golden Butterfly': PORTFOLIOS['Golden Butterfly'],
        'S&P 500': PORTFOLIOS['S&P 500'],
        'Nasdaq 100': PORTFOLIOS['Nasdaq 100'],
        'Semiconductors': PORTFOLIOS['Semiconductors']
    }
    
    # Collect all tickers
    all_tickers = [USD_ILS_TICKER]
    for tickers in portfolios.values():
        all_tickers.extend(tickers)
    all_tickers = list(set(all_tickers))
    
    print(f"Running DCA simulation from {start_year}...")
    print(f"Initial investment: {initial_investment:,} ILS")
    print(f"Monthly contribution: {monthly_investment:,} ILS (inflation-adjusted)")
    
    data = yf.download(all_tickers, start=f"{start_year}-01-01", progress=False)['Close']
    data = data.ffill().dropna()
    
    usd_ils = data[USD_ILS_TICKER]
    cpi = build_cpi_index(data.index)
    
    # Build portfolio price indices
    prices = pd.DataFrame(index=data.index)
    
    for name, tickers in portfolios.items():
        if len(tickers) == 1:
            prices[name] = data[tickers[0]] * usd_ils
        else:
            basket = data[tickers].multiply(usd_ils, axis=0)
            rets = basket.pct_change().mean(axis=1)
            price_series = (1 + rets).cumprod()
            price_series.iloc[0] = 1
            prices[name] = price_series * 100
    
    # Monthly resampling for DCA
    monthly_prices = prices.resample('ME').last()
    monthly_cpi = cpi.reindex(monthly_prices.index, method='ffill')
    
    # Simulate DCA
    results = {}
    
    for name in portfolios.keys():
        units = initial_investment / monthly_prices[name].iloc[0]
        total_invested = initial_investment
        
        for i in range(1, len(monthly_prices)):
            # Inflation-adjusted contribution
            adj_contribution = monthly_investment * (monthly_cpi.iloc[i] / monthly_cpi.iloc[0])
            units += adj_contribution / monthly_prices[name].iloc[i]
            total_invested += adj_contribution
        
        final_value = units * monthly_prices[name].iloc[-1]
        results[name] = {
            'invested': total_invested,
            'final_value': final_value,
            'return_pct': (final_value / total_invested - 1) * 100
        }
    
    # Print results
    print(f"\n--- DCA Simulation Results ---")
    for name, res in results.items():
        print(f"{name}:")
        print(f"  Total Invested: {res['invested']:,.0f} ILS")
        print(f"  Final Value: {res['final_value']:,.0f} ILS")
        print(f"  Return: {res['return_pct']:.2f}%")
    
    return results

# simulate_dca(monthly_investment=100, initial_investment=50000, start_year='2005')

In [ ]:
# @title Cell 15: Lump Sum vs DCA Strategy Comparison

def compare_lump_sum_vs_dca(lump_sum=50000, dca_total=80000, dca_months=24, start_year='2005'):
    """
    Compare lump sum investment vs DCA over specified period.
    """
    portfolios = {
        'S&P 500': PORTFOLIOS['S&P 500'],
        'Nasdaq 100': PORTFOLIOS['Nasdaq 100'],
        'Semiconductors': PORTFOLIOS['Semiconductors'],
        'Golden Butterfly': PORTFOLIOS['Golden Butterfly']
    }
    
    all_tickers = [USD_ILS_TICKER]
    for tickers in portfolios.values():
        all_tickers.extend(tickers)
    all_tickers = list(set(all_tickers))
    
    print(f"Comparing: {lump_sum:,} ILS lump sum vs {dca_total:,} ILS over {dca_months} months")
    
    data = yf.download(all_tickers, start=f"{start_year}-01-01", progress=False)['Close']
    data = data.ffill().dropna()
    
    usd_ils = data[USD_ILS_TICKER]
    cpi = build_cpi_index(data.index)
    
    # Build portfolio prices
    prices = pd.DataFrame(index=data.index)
    
    for name, tickers in portfolios.items():
        if len(tickers) == 1:
            prices[name] = data[tickers[0]] * usd_ils
        else:
            basket = data[tickers].multiply(usd_ils, axis=0)
            rets = basket.pct_change().mean(axis=1)
            p = (1 + rets).cumprod() * 100
            p.iloc[0] = 100
            prices[name] = p
    
    # Monthly prices
    monthly = prices.resample('ME').last()
    monthly_cpi = cpi.reindex(monthly.index, method='ffill')
    
    monthly_dca = dca_total / dca_months
    
    results = {}
    
    for name in portfolios.keys():
        # Lump sum
        lump_units = lump_sum / monthly[name].iloc[0]
        lump_final = lump_units * monthly[name].iloc[-1]
        
        # DCA
        dca_units = 0
        for i in range(min(dca_months, len(monthly))):
            dca_units += monthly_dca / monthly[name].iloc[i]
        dca_final = dca_units * monthly[name].iloc[-1]
        
        results[name] = {
            'lump_sum': lump_final,
            'dca': dca_final,
            'lump_return': (lump_final / lump_sum - 1) * 100,
            'dca_return': (dca_final / dca_total - 1) * 100
        }
    
    # Print results
    print(f"\n--- Strategy Comparison Results ({start_year}-Present) ---")
    print(f"{'Portfolio':<20} {'Lump Sum':>15} {'DCA':>15} {'Winner':>10}")
    print("-" * 60)
    
    for name, res in results.items():
        winner = 'Lump Sum' if res['lump_return'] > res['dca_return'] else 'DCA'
        print(f"{name:<20} {res['lump_return']:>14.2f}% {res['dca_return']:>14.2f}% {winner:>10}")
    
    return results

# compare_lump_sum_vs_dca(lump_sum=50000, dca_total=80000, dca_months=24, start_year='2005')

In [ ]:
# @title Cell 16: Yearly Returns Breakdown Table

def print_yearly_breakdown(start_year='2007'):
    """
    Print yearly returns for all portfolios.
    """
    portfolios = {
        'Golden Butterfly': PORTFOLIOS['Golden Butterfly'],
        'Ivy Portfolio': PORTFOLIOS['Ivy Portfolio'],
        'S&P 500': PORTFOLIOS['S&P 500'],
        'Nasdaq 100': PORTFOLIOS['Nasdaq 100']
    }
    
    all_tickers = [USD_ILS_TICKER]
    for tickers in portfolios.values():
        all_tickers.extend(tickers)
    all_tickers = list(set(all_tickers))
    
    print("Loading data and calculating yearly returns...")
    data = yf.download(all_tickers, start=f"{start_year}-01-01", progress=False)['Close']
    data = data.ffill().dropna()
    
    usd_ils = data[USD_ILS_TICKER]
    cpi = build_cpi_index(data.index)
    
    # Calculate daily returns for each portfolio
    portfolio_values = pd.DataFrame(index=data.index)
    
    for name, tickers in portfolios.items():
        if len(tickers) == 1:
            prices_ils = data[tickers[0]] * usd_ils
            portfolio_values[name] = prices_ils
        else:
            basket_ils = data[tickers].multiply(usd_ils, axis=0)
            daily_rets = basket_ils.pct_change().mean(axis=1)
            portfolio_values[name] = (1 + daily_rets).cumprod() * 100
    
    # Calculate yearly returns
    yearly = portfolio_values.resample('YE').last()
    yearly_cpi = cpi.resample('YE').last()
    
    yearly_returns = yearly.pct_change() * 100
    yearly_inflation = yearly_cpi.pct_change() * 100
    
    # Print table
    print(f"\n--- Yearly Returns (ILS, Nominal) ---")
    print(f"{'Year':<6}", end='')
    for name in portfolios.keys():
        print(f"{name:>18}", end='')
    print(f"{'Inflation':>12}")
    print("-" * 80)
    
    for year in yearly_returns.index[1:]:
        print(f"{year.year:<6}", end='')
        for name in portfolios.keys():
            val = yearly_returns.loc[year, name]
            print(f"{val:>17.2f}%", end='')
        inf = yearly_inflation.loc[year] if year in yearly_inflation.index else 0
        print(f"{inf:>11.2f}%")
    
    return yearly_returns

# print_yearly_breakdown('2007')

In [ ]:
# @title Cell 17: Master Simulation - Full Report

def run_master_simulation():
    """
    Run comprehensive analysis with all data and visualizations.
    """
    print("=" * 60)
    print("MASTER SIMULATION: Complete Portfolio Analysis")
    print("=" * 60)
    
    portfolios = {
        'S&P 500': PORTFOLIOS['S&P 500'],
        'Nasdaq 100': PORTFOLIOS['Nasdaq 100'],
        'Semiconductors': PORTFOLIOS['Semiconductors'],
        'Golden Butterfly': PORTFOLIOS['Golden Butterfly']
    }
    
    all_tickers = [USD_ILS_TICKER]
    for tickers in portfolios.values():
        all_tickers.extend(tickers)
    all_tickers = list(set(all_tickers))
    
    print("\nDownloading data (including dividends)...")
    data = yf.download(all_tickers, start="2004-12-01", progress=False, auto_adjust=True)['Close']
    data = data.ffill().dropna()
    
    usd_ils = data[USD_ILS_TICKER]
    
    # Build geometric daily CPI
    daily_factors = pd.Series(index=data.index, dtype=float)
    for year in sorted(INFLATION_DATA.keys()):
        days_in_year = data.index[data.index.year == year]
        if len(days_in_year) > 0:
            annual_rate = INFLATION_DATA[year] / 100
            daily_rate = (1 + annual_rate) ** (1 / len(days_in_year))
            daily_factors.loc[days_in_year] = daily_rate
    
    daily_factors = daily_factors.fillna(1.0)
    cpi_curve = daily_factors.cumprod() * 100
    
    # Normalize to Jan 1, 2005 = 100
    base_date = pd.Timestamp("2005-01-01")
    valid_idx = cpi_curve.index.searchsorted(base_date)
    if valid_idx < len(cpi_curve):
        base_value = cpi_curve.iloc[valid_idx]
        cpi_curve = cpi_curve / base_value * 100
    
    # Calculate portfolio prices
    prices_ils = pd.DataFrame(index=data.index)
    real_prices = pd.DataFrame(index=data.index)
    
    for name, tickers in portfolios.items():
        if len(tickers) == 1:
            p = data[tickers[0]] * usd_ils
            norm_p = (p / p.iloc[0]) * 100
        else:
            basket = data[tickers].multiply(usd_ils, axis=0)
            rets = basket.pct_change().mean(axis=1)
            norm_p = (1 + rets).cumprod() * 100
            norm_p.iloc[0] = 100
        
        prices_ils[name] = norm_p
        real_prices[name] = norm_p / cpi_curve * 100
    
    # Print summary
    print(f"\n--- Final Results (2005-Present) ---")
    print(f"{'Portfolio':<20} {'Nominal':>12} {'Real':>12}")
    print("-" * 45)
    
    for name in portfolios.keys():
        nom = prices_ils[name].iloc[-1] - 100
        real = real_prices[name].iloc[-1] - 100
        print(f"{name:<20} {nom:>11.2f}% {real:>11.2f}%")
    
    print(f"\nTotal Inflation: {cpi_curve.iloc[-1] - 100:.2f}%")
    
    # Plot
    colors = {
        'S&P 500': 'blue',
        'Nasdaq 100': 'black',
        'Semiconductors': 'purple',
        'Golden Butterfly': '#D4AF37'
    }
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Plot 1: Nominal returns
    ax1 = axes[0, 0]
    for name in portfolios.keys():
        ax1.plot(prices_ils.index, prices_ils[name], label=name, color=colors[name], linewidth=2)
    ax1.axhline(y=100, color='gray', linestyle='--', linewidth=1)
    ax1.set_title('Nominal Returns (ILS)', fontsize=12)
    ax1.set_ylabel('Value (Base=100)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Real returns
    ax2 = axes[0, 1]
    for name in portfolios.keys():
        ax2.plot(real_prices.index, real_prices[name], label=name, color=colors[name], linewidth=2)
    ax2.axhline(y=100, color='gray', linestyle='--', linewidth=1)
    ax2.set_title('Real Returns (Inflation-Adjusted)', fontsize=12)
    ax2.set_ylabel('Value (Base=100)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Log scale nominal
    ax3 = axes[1, 0]
    for name in portfolios.keys():
        ax3.plot(prices_ils.index, prices_ils[name], label=name, color=colors[name], linewidth=2)
    ax3.set_yscale('log')
    ax3.axhline(y=100, color='gray', linestyle='--', linewidth=1)
    ax3.set_title('Nominal Returns (Log Scale)', fontsize=12)
    ax3.set_ylabel('Value (Base=100, Log)')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # Plot 4: CPI
    ax4 = axes[1, 1]
    ax4.plot(cpi_curve.index, cpi_curve, label='CPI (Inflation)', color='red', linewidth=2)
    ax4.axhline(y=100, color='gray', linestyle='--', linewidth=1)
    ax4.set_title('Israeli CPI Index', fontsize=12)
    ax4.set_ylabel('Index Value')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return {'nominal': prices_ils, 'real': real_prices, 'cpi': cpi_curve}

# run_master_simulation()

In [ ]:
# @title Cell 18: Quick Run - Execute All Analyses

def run_all_analyses():
    """
    Run all main analyses in sequence.
    """
    print("Running complete analysis suite...\n")
    
    # 1. Silver analysis (1 year)
    print("\n" + "="*60)
    print("1. SILVER ANALYSIS (1 Year)")
    print("="*60)
    analyze_asset_returns('SLV', 'Silver', '1y')
    
    # 2. S&P 500 synthetic fund
    print("\n" + "="*60)
    print("2. S&P 500 SYNTHETIC ISRAELI FUND (5 Years)")
    print("="*60)
    analyze_sp500_synthetic_il()
    
    # 3. Portfolio comparison
    print("\n" + "="*60)
    print("3. GOLDEN BUTTERFLY vs IVY PORTFOLIO (20 Years)")
    print("="*60)
    compare_portfolios_shekel_20y()
    
    # 4. Multi-asset comparison
    print("\n" + "="*60)
    print("4. ALL PORTFOLIOS COMPARISON (2015-Present)")
    print("="*60)
    compare_all_portfolios('2015')
    
    # 5. Master simulation
    print("\n" + "="*60)
    print("5. MASTER SIMULATION (2005-Present)")
    print("="*60)
    run_master_simulation()
    
    print("\n" + "="*60)
    print("ANALYSIS COMPLETE!")
    print("="*60)

# Uncomment to run all analyses:
# run_all_analyses()